# 1.Install And Import Libraries

In [ ]:
#!pip install transformers datasets seqeval -q

In [ ]:
#!pip install evaluate

In [1]:
import pandas as pd
import numpy as np
import re
import string
import os
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import warnings
from datasets import load_dataset
import evaluate

# Task 1: Dataset Selection

In [33]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer)
dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")
label_list = dataset["train"].features["chunk_tags"].feature.names
print(label_list)

['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


# Task 2: Data Preprocessing

In [3]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
def tokenize_and_align_labels(examples):
    # Tokenize words using BERT tokenizer
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Handle special tokens with -100 so they are ignored in the loss function
            if word_idx is None:
                label_ids.append(-100)
            # Assign the label to the first token of a given word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # Handle subwords: assign -100 to subsequent subword tokens
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply preprocessing to the entire dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


# Task 3: Model Setup

In [5]:
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

In [6]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
print(model.config.num_labels)

23


# Task 5.Evaluation Setup 

In [8]:
seqeval = evaluate.load("seqeval")

In [9]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove -100 from labels and predictions before evaluation
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Task 4 Training

In [20]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    #report_to="none",              # Disables reporting to external tools
    #disable_tqdm=True,             # Disables the progress bar tracker             # Disables the progress bar tracker
)

from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].select(range(1000)), # Subset for faster testing
    eval_dataset=tokenized_datasets["validation"].select(range(500)),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.361177,0.844697,0.822271,0.833333,0.913860
2,No log,0.332639,0.851810,0.832965,0.842282,0.920702
3,No log,0.329525,0.853640,0.834440,0.843931,0.921930


C:\Users\bhanu\anaconda3\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\bhanu\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\bhanu\anaconda3\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=189, training_loss=0.3011218681537285, metrics={'train_runtime': 950.8789, 'train_samples_per_second': 3.155, 'train_steps_per_second': 0.199, 'total_flos': 32685888589200.0, 'train_loss': 0.3011218681537285, 'epoch': 3.0})

In [19]:
#Run Evaluation
trainer.evaluate()

{'eval_loss': '0.3295', 'eval_model_preparation_time': '0.0047', 'eval_precision': '0.8536', 'eval_recall': '0.8344', 'eval_f1': '0.8439', 'eval_accuracy': '0.9219', 'eval_runtime': '39.69', 'eval_samples_per_second': '12.6', 'eval_steps_per_second': '0.806', 'epoch': 0}


{'eval_loss': 0.3295247554779053,
 'eval_model_preparation_time': 0.0047,
 'eval_precision': 0.8536401357978122,
 'eval_recall': 0.8344395280235988,
 'eval_f1': 0.8439306358381503,
 'eval_accuracy': 0.9219298245614035,
 'eval_runtime': 39.6874,
 'eval_samples_per_second': 12.598,
 'eval_steps_per_second': 0.806,
 'epoch': 0}

# Task 6: Inference 10%

In [32]:
def predict_sentence(sentence):
    tokens = sentence.split()

    inputs = tokenizer(
        tokens,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True)

    outputs = model(**inputs)
    predictions = outputs.logits.argmax(dim=2)

    word_ids = inputs.word_ids()

    predicted_labels = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue

        if word_idx != previous_word_idx:
            label_id = predictions[0][idx].item()
            predicted_labels.append(label_list[label_id])

        previous_word_idx = word_idx

    return list(zip(tokens, predicted_labels))

In [30]:
sentence = "John works at Google in California"
print(predict_sentence(sentence))

[('John', 'B-NP'), ('works', 'B-VP'), ('at', 'B-PP'), ('Google', 'B-NP'), ('in', 'B-PP'), ('California', 'B-NP')]


# Task 7.Comparison

# Differences Between POS Tagging and Chunking

## POS Tagging: Assigns grammatical labels (noun, verb, adjective) to each word → works at word level and is easier.
## Chunking: Groups words into phrases (noun phrase, verb phrase) → works at phrase level and is slightly more complex.
## *Key Difference:* POS tagging identifies individual word roles, while chunking identifies meaningful word groups.

# Task 8.blog

## Differences:
POS Tagging focuses on labeling each word with its grammatical role, whereas Chunking groups words into phrases to give more structural meaning to a sentence.

## Challenges Faced:
Handling subword tokenization and aligning labels with tokens was difficult,
especially when words split into multiple subwords.
## Handling Special Tokens:
Special tokens such as [CLS] and [SEP], along with additional subword tokens, do not correspond to actual labels. Therefore, assigning a value of -100 to these tokens was necessary so that they are ignored during loss computation by the model.
# Word Tokenization:
Transformer models like BERT use WordPiece tokenization, which breaks unknown or rare words into smaller subword units. The main challenge was ensuring correct alignment between original word-level labels and tokenized subwords.

## Observations:
Chunking provides better context understanding than POS tagging, but POS tagging is faster and simpler to implement.